# Vertex reconstruction efficiency with finer ΔR bins

LL, LR, RR, and total efficiencies use the existing selections and common generated denominator. LR includes electron L and positron R only. Reverse RL remains excluded.

Run with the **LPC kernel**, using `drEff_cont.coffea`. There are 18 ΔR intervals. The numerical-axis curve places each interval's efficiency at its arithmetic midpoint and shows its extent with horizontal bars. Connecting lines guide the eye, rather than giving a fitted or unbinned efficiency.

Heatmaps appear in pages of at most four panels, covering all intervals. Counts are summed before division. Underflow and overflow remain excluded. The final cell explicitly saves figures on LPC in a fresh folder. No files are saved by the earlier plotting cells.


In [1]:
from pathlib import Path

LPC_REPO = Path('/uscms/home/reshmar/nobackup/IDMe_Run3_Collab/CMSSW_13_0_13/src/iDMe')
TOOLS_DIR = LPC_REPO / 'python_analysis/analysisTools'

%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt
from coffea import util
from coffea.processor import accumulate
import os
import re
import pandas as pd
import hist
import matplotlib as mpl
import mplhep
import sys
import hist
import hist.intervals


sys.path.append("/uscms/home/reshmar/nobackup/IDMe_Run3_Collab/CMSSW_13_0_13/src/iDMe/python_analysis/analysisTools")
import plotTools as ptools
import utils

## Load the new output and select samples
The same sample selection and stored cut apply throughout. Check that all four histogram categories have the same new ΔR edges.


In [2]:
f_sig = LPC_REPO / 'python_analysis/coffea/VertexRes2026/drEff_cont.coffea'
sig_histo = util.load(str(f_sig))[0]

samples = {
    10: 
    {

        0.2: [100],
    },
    20: 
    {

        0.2: [100],

    },
    30: 
    {

        0.2: [100]
    },
    40: 
    {
        0.2: [100],
    },
    50:
    {

        0.2: [100],
    },
    60:
    {

         0.1: [100],
         0.2: [100]

    },
    70:
    {    0.1: [100],
         0.2: [100],
    },
    80:
    {    0.1: [100],
         0.2: [100],
    },
    90:
    {    0.1: [100],
         0.2: [100],
    },
    100:
    {    
         0.2: [100],
    }
}


cut = 'cut0'
year = 2022

categories = ('LL', 'LR', 'RR')
required_histograms = ['genvtx_pt_vxy_dr_den'] + [
    f'genvtx_pt_vxy_dr_{category}' for category in categories
]

si = utils.get_signal_point_dict(sig_histo)
selected_points = []
for m1, deltas in samples.items():
    for delta, ctaus in deltas.items():
        for ctau in ctaus:
            selected = si[
                np.isclose(si.m1, m1, rtol=1e-8, atol=1e-8)
                & np.isclose(si.delta, delta, rtol=1e-8, atol=1e-8)
                & (si.ctau == ctau)
            ]
            if len(selected) != 1:
                raise ValueError(f'Expected one sample for {(m1, delta, ctau)}; found {len(selected)}.')
            sample_name = selected['name'].iloc[0]
            if sample_name in [point[3] for point in selected_points]:
                raise ValueError(f'Duplicate sample selection: {sample_name}')
            selected_points.append((m1, delta, ctau, sample_name))

if not selected_points:
    raise ValueError('Choose at least one signal point.')

# Check axis names, order, and edges across every histogram and sample.
reference_edges = None
for m1, delta, ctau, sample_name in selected_points:
    for suffix in ('den',) + categories:
        h = sig_histo[f'genvtx_pt_vxy_dr_{suffix}'][{'samp': sample_name, 'cut': cut}]
        expected_names = (f'pt_{suffix}', f'vxy_{suffix}', f'dr_{suffix}')
        if tuple(axis.name for axis in h.axes) != expected_names:
            raise ValueError(f'Unexpected axes for {suffix}: {h.axes}')
        current_edges = tuple(np.asarray(axis.edges).copy() for axis in h.axes)
        if reference_edges is None:
            reference_edges = current_edges
        else:
            for expected, actual in zip(reference_edges, current_edges):
                np.testing.assert_array_equal(expected, actual)

pt_edges, vxy_edges, dr_edges = reference_edges
expected_dr_edges = np.asarray([
    0, 0.01, 0.02, 0.04, 0.06, 0.08, 0.1,
    0.15, 0.2, 0.3, 0.4, 0.5, 0.7, 1, 1.5, 2, 3, 4, 5
])
np.testing.assert_allclose(dr_edges, expected_dr_edges, rtol=0, atol=1e-12)
print('Number of ΔR bins:', len(dr_edges) - 1)
print('Selected samples:', [point[3] for point in selected_points])
print('ΔR edges:', dr_edges)


Number of ΔR bins: 18
Selected samples: ['sig_2022_Mchi-11p0_dMchi-2p0_ctau-100', 'sig_2022_Mchi-22p0_dMchi-4p0_ctau-100', 'sig_2022_Mchi-33p0_dMchi-6p0_ctau-100', 'sig_2022_Mchi-44p0_dMchi-8p0_ctau-100', 'sig_2022_Mchi-55p0_dMchi-10p0_ctau-100', 'sig_2022_Mchi-63p0_dMchi-6p0_ctau-100', 'sig_2022_Mchi-66p0_dMchi-12p0_ctau-100', 'sig_2022_Mchi-73p5_dMchi-7p0_ctau-100', 'sig_2022_Mchi-77p0_dMchi-14p0_ctau-100', 'sig_2022_Mchi-84p0_dMchi-8p0_ctau-100', 'sig_2022_Mchi-88p0_dMchi-16p0_ctau-100', 'sig_2022_Mchi-94p5_dMchi-9p0_ctau-100', 'sig_2022_Mchi-99p0_dMchi-18p0_ctau-100', 'sig_2022_Mchi-110p0_dMchi-20p0_ctau-100']
ΔR edges: [0.   0.01 0.02 0.04 0.06 0.08 0.1  0.15 0.2  0.3  0.4  0.5  0.7  1.
 1.5  2.   3.   4.   5.  ]


## Collect counts using the plotTools function
The function returns 2D counts after selecting a ΔR bin. Its intermediate count figures are closed; the efficiency figures are displayed below. No rebinning is applied, and underflow/overflow bins are excluded from the maps.


In [ ]:
def collect_counts(suffix, dr_bin):
    variable = f'genvtx_pt_vxy_dr_{suffix}'
    summed = None
    for m1, delta, ctau, sample_name in selected_points:
        fig, ax = plt.subplots(figsize=(7, 7))
        plot_dict = {'variable': variable, 'cut': cut, 'year': year}
        style_dict = {
            'fig': fig, 'ax': ax,
            'xrebin': None, 'yrebin': None,
            'xlim': None, 'ylim': None,
            'doLogx': False, 'doLogy': False, 'doLogz': False,
            'xlabel': 'Gen vertex pT [GeV]',
            'ylabel': 'Gen vertex vxy [cm]',
            'zlabel': 'Entries', 'flow': None, 'doSave': False,
        }
        try:
            counts, returned_pt, returned_vxy = ptools.plot_signal_2D_dr(
                sig_histo, m1, delta, ctau, plot_dict, style_dict,
                dr_bin=dr_bin,
            )
            counts = np.asarray(counts, dtype=float)
        finally:
            plt.close(fig)
        np.testing.assert_array_equal(pt_edges, returned_pt)
        np.testing.assert_array_equal(vxy_edges, returned_vxy)
        if counts.shape != (len(pt_edges) - 1, len(vxy_edges) - 1):
            raise ValueError('Unexpected count-array shape.')
        summed = counts.copy() if summed is None else summed + counts
    return summed


def efficiency_and_uncertainty(numerator, denominator):
    if not np.all(np.isfinite(numerator)) or not np.all(np.isfinite(denominator)):
        raise ValueError('Counts contain non-finite values.')
    if np.any(numerator < 0) or np.any(denominator < 0):
        raise ValueError('This efficiency calculation requires nonnegative counts.')
    if np.any(numerator > denominator):
        raise ValueError('Numerator exceeds denominator. Check the selections.')
    if not np.allclose(numerator, np.round(numerator)) or not np.allclose(denominator, np.round(denominator)):
        raise ValueError('Binomial uncertainties here require unit-weight event counts.')
    valid = denominator > 0
    efficiency = np.divide(
        numerator, denominator,
        out=np.full_like(numerator, np.nan, dtype=float), where=valid,
    )
    uncertainty = np.full((2,) + numerator.shape, np.nan)
    if np.any(valid):
        uncertainty[:, valid] = hist.intervals.ratio_uncertainty(
            numerator[valid], denominator[valid], 'efficiency'
        )
    return efficiency, uncertainty


results = {}
for dr_bin in range(len(dr_edges) - 1):
    denominator = collect_counts('den', dr_bin)
    entry = {
        'denominator': denominator,
        'dr_range': (dr_edges[dr_bin], dr_edges[dr_bin + 1]),
    }
    for category in categories:
        numerator = collect_counts(category, dr_bin)
        efficiency, uncertainty = efficiency_and_uncertainty(numerator, denominator)
        entry[category] = {
            'numerator': numerator,
            'efficiency': efficiency,
            'uncertainty': uncertainty,
        }
    total_numerator = sum(entry[category]['numerator'] for category in categories)
    total_efficiency, total_uncertainty = efficiency_and_uncertainty(total_numerator, denominator)
    entry['Total'] = {
        'numerator': total_numerator,
        'efficiency': total_efficiency,
        'uncertainty': total_uncertainty,
    }
    np.testing.assert_allclose(
        total_efficiency,
        sum(entry[category]['efficiency'] for category in categories),
        equal_nan=True,
    )
    results[dr_bin] = entry
    low, high = entry['dr_range']
    print(f'Calculated LL, LR, RR, and total for {low:g} ≤ ΔR < {high:g}')


Calculated LL, LR, RR, and total for 0 ≤ ΔR < 0.01
Calculated LL, LR, RR, and total for 0.01 ≤ ΔR < 0.02
Calculated LL, LR, RR, and total for 0.02 ≤ ΔR < 0.04
Calculated LL, LR, RR, and total for 0.04 ≤ ΔR < 0.06
Calculated LL, LR, RR, and total for 0.06 ≤ ΔR < 0.08


## Heatmaps for every ΔR bin
Each category has five pages: four pages with four panels and a final page with two panels. Blank cells have zero denominator entries. All maps use the same colour range, 0 to 1.


In [ ]:
def iter_efficiency_figures(category):
    """Yield one heatmap page at a time, without saving or showing it."""
    ordered_bins = sorted(results)
    for start in range(0, len(ordered_bins), 4):
        page_bins = ordered_bins[start:start + 4]
        rows = (len(page_bins) + 1) // 2
        fig, axes = plt.subplots(
            rows, 2, figsize=(15, 6.5 * rows),
            layout='constrained', squeeze=False,
        )
        active_axes = []
        for ax, dr_bin in zip(axes.flat, page_bins):
            active_axes.append(ax)
            efficiency = results[dr_bin][category]['efficiency'].T
            n_vxy, n_pt = efficiency.shape
            pt_centers = np.arange(n_pt) + 0.5
            vxy_centers = np.arange(n_vxy) + 0.5
            mesh = ax.pcolormesh(
                np.arange(n_pt + 1), np.arange(n_vxy + 1),
                np.ma.masked_invalid(efficiency),
                cmap='viridis', shading='flat', vmin=0, vmax=1,
            )
            ax.set_aspect('equal')
            for iy, y in enumerate(vxy_centers):
                for ix, x in enumerate(pt_centers):
                    value = efficiency[iy, ix]
                    if np.isfinite(value):
                        ax.text(x, y, f'{100 * value:.2f}%', ha='center', va='center',
                                fontsize=10, color='white' if value < 0.5 else 'black')
            ax.set_xticks(pt_centers)
            ax.set_xticklabels([f'{a:g}–{b:g}' for a,b in zip(pt_edges[:-1],pt_edges[1:])])
            ax.set_yticks(vxy_centers)
            ax.set_yticklabels([f'{a:g}–{b:g}' for a,b in zip(vxy_edges[:-1],vxy_edges[1:])])
            ax.set_xlabel(r'Gen $p_T(e^+e^-)$ bin [GeV]')
            ax.set_ylabel(r'Gen $L_{xy}(e^+e^-)$ bin [cm]')
            low, high = results[dr_bin]['dr_range']
            ax.set_title(f'{category}: {low:g} ≤ Gen ΔR < {high:g}')
        for ax in list(axes.flat)[len(page_bins):]:
            ax.set_visible(False)
        page = start // 4 + 1
        total_pages = (len(ordered_bins) + 3) // 4
        label = 'Total = LL + LR + RR (reverse RL excluded)' if category=='Total' else f'{category} vertex reconstruction efficiency'
        fig.suptitle(f'{label} — page {page}/{total_pages}')
        fig.colorbar(mesh, ax=active_axes, label=f'{category} efficiency', shrink=0.8)
        yield page, page_bins, fig


def plot_efficiency_category(category):
    for page, page_bins, figure in iter_efficiency_figures(category):
        print(f'{category}: page {page}, ΔR bin indices {page_bins}')
        try:
            plt.show()
        finally:
            plt.close(figure)


## LL efficiency — all 18 ΔR bins


## LR efficiency — all 18 ΔR bins


## RR efficiency — all 18 ΔR bins


## Total efficiency — all 18 ΔR bins
Total = (N_LL + N_LR + N_RR) / N_den. Its uncertainty uses the summed numerator, rather than adding category uncertainties. Reverse RL is excluded.


## Numerical ΔR curves and checks
The next cells print counts and uncertainties for all 18 intervals and draw LL, RR, and LR on the same numerical ΔR axis. This preserves the existing sample, pT, and vxy selections. Undefined efficiencies remain gaps.


In [ ]:
dr_bins_ordered = sorted(results)
categories = ("LL", "RR", "LR")

denominator_vs_dr = np.asarray([
    results[i]["denominator"].sum()
    for i in dr_bins_ordered
], dtype=float)

efficiencies_vs_dr = {}

print("CALCULATION SETTINGS")
print("Signal points:", samples)
print("ΔR bin order:", dr_bins_ordered)
print(f"Included pT: [{pt_edges[0]:g}, {pt_edges[-1]:g}) GeV")
print(f"Included vxy: [{vxy_edges[0]:g}, {vxy_edges[-1]:g}) cm")
print("Underflow and overflow are excluded.")
print("Method: sum numerator counts / sum denominator counts.")
print("LR includes electron-L/positron-R only.")

for category in categories:

    numerator_vs_dr = np.asarray([
        results[i][category]["numerator"].sum()
        for i in dr_bins_ordered
    ], dtype=float)

    efficiency_vs_dr, uncertainty_vs_dr = (
        efficiency_and_uncertainty(
            numerator_vs_dr,
            denominator_vs_dr,
        )
    )

    # Independently check the returned efficiency against N / D.
    direct_ratio = np.divide(
        numerator_vs_dr,
        denominator_vs_dr,
        out=np.full_like(numerator_vs_dr, np.nan),
        where=denominator_vs_dr > 0,
    )

    np.testing.assert_allclose(
        efficiency_vs_dr,
        direct_ratio,
        equal_nan=True,
    )

    summary = pd.DataFrame({
        "dr_bin": dr_bins_ordered,
        "dr_low": [
            results[i]["dr_range"][0]
            for i in dr_bins_ordered
        ],
        "dr_high": [
            results[i]["dr_range"][1]
            for i in dr_bins_ordered
        ],
        "numerator_count": numerator_vs_dr,
        "denominator_count": denominator_vs_dr,
        "efficiency": efficiency_vs_dr,
        "efficiency_percent": 100 * efficiency_vs_dr,
        "uncertainty_low": uncertainty_vs_dr[0],
        "uncertainty_high": uncertainty_vs_dr[1],
    })

    efficiencies_vs_dr[category] = {
        "numerator": numerator_vs_dr,
        "efficiency": efficiency_vs_dr,
        "uncertainty": uncertainty_vs_dr,
        "table": summary,
    }

    print(f"\n{category}: COUNTS AND CALCULATIONS")

    for position, dr_bin in enumerate(dr_bins_ordered):
        low, high = results[dr_bin]["dr_range"]
        numerator = numerator_vs_dr[position]
        denominator = denominator_vs_dr[position]

        num_shape = results[dr_bin][category]["numerator"].shape
        den_shape = results[dr_bin]["denominator"].shape

        if num_shape != den_shape:
            raise ValueError(
                f"{category}, ΔR bin {dr_bin}: "
                "numerator and denominator shapes differ."
            )

        print(
            f"Bin {dr_bin}: [{low:g}, {high:g}), "
            f"count-array shape = {num_shape}"
        )

        if denominator > 0:
            eff = efficiency_vs_dr[position]
            unc_low = uncertainty_vs_dr[0, position]
            unc_high = uncertainty_vs_dr[1, position]

            print(
                f"  {numerator:g} / {denominator:g} "
                f"= {eff:.6f} = {100 * eff:.3f}%"
            )
            print(
                f"  Uncertainty: -{unc_low:.6f}, "
                f"+{unc_high:.6f}"
            )
        else:
            print(
                f"  Numerator = {numerator:g}, denominator = 0. "
                "Efficiency undefined; no point will be plotted."
            )

    print(f"\n{category}: SUMMARY TABLE")
    print(summary.to_string(index=False))
    print("Check passed: efficiencies equal summed numerator / denominator.")

# These categories are mutually exclusive in drEff.py.
summed_category_counts = sum(
    efficiencies_vs_dr[category]["numerator"]
    for category in categories
)

if np.any(summed_category_counts > denominator_vs_dr):
    raise ValueError(
        "LL + RR + LR counts exceed the denominator. "
        "Check the histogram selections."
    )

print("\nCheck passed: LL + RR + LR counts do not exceed the denominator.")

In [ ]:
# Numerical ΔR positions rather than equally spaced interval categories.
dr_low = np.asarray([results[i]['dr_range'][0] for i in dr_bins_ordered])
dr_high = np.asarray([results[i]['dr_range'][1] for i in dr_bins_ordered])
x_positions = (dr_low + dr_high) / 2
x_errors = np.vstack((x_positions - dr_low, dr_high - x_positions))

print('NUMERICAL X-AXIS CHECK')
for i, low, high, x in zip(dr_bins_ordered, dr_low, dr_high, x_positions):
    print(f'Bin {i}: [{low:g}, {high:g}), plotted midpoint = {x:g}')
print('Horizontal bars show bin widths. Lines connect measured bin efficiencies only.')


def plot_efficiency_vs_dr():
    colours = {'LL':'tab:blue', 'RR':'tab:green', 'LR':'tab:orange'}
    markers = {'LL':'o', 'RR':'s', 'LR':'^'}
    fig, ax = plt.subplots(figsize=(11, 7))
    for category in ('LL','RR','LR'):
        efficiency = efficiencies_vs_dr[category]['efficiency']
        uncertainty = efficiencies_vs_dr[category]['uncertainty']
        valid = np.isfinite(efficiency)
        print(f'PLOTTING {category}: {valid.sum()}/{len(valid)} intervals')
        for j in range(len(dr_bins_ordered)):
            if valid[j]:
                print(f'  ΔR [{dr_low[j]:g}, {dr_high[j]:g}), x={x_positions[j]:g}: '
                      f'eff={efficiency[j]:.6f}, '
                      f'error=-{uncertainty[0,j]:.6f}/+{uncertainty[1,j]:.6f}')
            else:
                print(f'  ΔR [{dr_low[j]:g}, {dr_high[j]:g}): no denominator entries')
        ax.plot(x_positions, efficiency, color=colours[category], linewidth=1.5)
        ax.errorbar(
            x_positions[valid], efficiency[valid],
            xerr=x_errors[:,valid], yerr=uncertainty[:,valid],
            fmt=markers[category], markersize=5, capsize=3,
            color=colours[category], label=category,
        )
    ax.set_xlim(dr_low[0], dr_high[-1])
    ax.set_ylim(bottom=0)
    ax.set_xlabel(r'Gen $\Delta R(e^+e^-)$')
    ax.set_ylabel('Vertex reconstruction efficiency')
    ax.set_title('LL, RR, and LR efficiency vs ΔR')
    ax.grid(alpha=0.3)
    ax.legend(title='Category')
    fig.tight_layout()
    return fig, ax

# Linear numerical axis. To examine small ΔR more closely, change the
# displayed range with ax.set_xlim(0, 0.2) before plt.show(), then rerun.
combined_figure, ax = plot_efficiency_vs_dr()
plt.show()


In [ ]:
# SAVE ALL HEATMAP PAGES AND THE COMBINED ΔR CURVE ON LPC.
# This redraws heatmaps from existing arrays, without rerunning the analysis.
from datetime import datetime

if 'combined_figure' not in globals():
    raise RuntimeError('Run the numerical ΔR curve cell first.')
output_parent = LPC_REPO / 'python_analysis/plots/VertexRecEff_vs_dR_finebins'
output_parent.mkdir(parents=True, exist_ok=True)
output_dir = output_parent / datetime.now().strftime('%Y%m%d_%H%M%S_%f')
output_dir.mkdir(exist_ok=False)
saved_files = []


def save_figure(figure, name):
    for extension in ('png','pdf'):
        destination = output_dir / f'{name}.{extension}'
        with destination.open('xb') as stream:
            figure.savefig(stream, format=extension, dpi=300,
                           bbox_inches='tight', facecolor='white')
        saved_files.append(destination)
        print(f'Saved: {destination}')


save_figure(combined_figure, 'LL_RR_LR_efficiency_vs_dR_numerical')
for category in ('LL','LR','RR','Total'):
    for page, page_bins, figure in iter_efficiency_figures(category):
        try:
            name = f'{category}_efficiency_page{page:02d}_bins{page_bins[0]:02d}-{page_bins[-1]:02d}'
            save_figure(figure, name)
        finally:
            plt.close(figure)

expected_figures = 4 * ((len(results) + 3) // 4) + 1
assert len(saved_files) == 2 * expected_figures
print(f'\nSaved {expected_figures} figures as PNG and PDF ({len(saved_files)} files).')
print(f'LPC output folder: {output_dir}')
